# Single-year vs multi-year ICESat-2 training: which AGBD model is better, and why?

**Question.** The v3.1 AGBD maps can be built from rF models trained on ICESat-2 ATL08 30 m observations from **one year** (the target year, e.g. 2020), or from a **multi-year** pool (2019–2024, each observation paired with HLS from its own year: "multi-year correspondence"). Which strategy gives the better map?

**How this notebook answers it.** Four independent lines of evidence, each ending in a one-line takeaway, then a scorecard (§6) that states the verdict and how strong it is.

| § | Evidence | What it asks | Role in the verdict |
|---|---|---|---|
| 2 | **A. Training data** | How many observations does each model learn from, how evenly do they cover the tile, and when were they acquired? | Supporting (explains *why*) |
| 3 | **B. Map agreement** | Where do the two 2020 maps disagree, and does disagreement follow the training-data differences? | Descriptive (where it matters) |
| 4 | **C. Accuracy vs field plots** | Against the same screened field plots, which 2020 map is closer to the reference? | **Primary** (*who is right*) |
| 5 | **D. Temporal stability** | In mature, undisturbed stands, which time series has less year-to-year noise? | Supporting |

All differences are reported as **multi − single** with 95 % bootstrap confidence intervals. A criterion only counts as a win when its CI excludes zero; otherwise it is reported as "no clear difference".

**Jump to §6 for the verdict.** Each section's text explains how to read its figure.

*Why a new notebook.* `compare_icesat2_multi_single_yr_models.ipynb` compared the spatial pattern of the two training sets with Ripley's L-function. ICESat-2 samples along ground tracks, so both sets are strongly clustered by construction. L(r) then mostly reflects track geometry and sample size rather than sampling quality, so it cannot say which model is better (it rated the smaller single-year set "better"). It also never compared the resulting maps against independent reference data. Here the training data are summarised with directly interpretable metrics (obs per tile, share of the tile covered, distance to the nearest training obs, acquisition timing), and the maps themselves are judged against field plots.

In [ ]:
suppressPackageStartupMessages({
    library(sf)
    library(terra)
    library(dplyr)
    library(tidyr)
    library(purrr)
    library(readr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
    library(scales)
})
if (packageVersion("ggplot2") >= "3.5.0" && packageVersion("patchwork") < "1.2.0")
    stop("patchwork ", packageVersion("patchwork"), " cannot combine ggplot2 >= 3.5 plots. ",
         "Install patchwork 1.2.0 (remotes::install_version('patchwork', '1.2.0')) and restart the kernel.")
sf_use_s2(FALSE)
options(dplyr.summarise.inform = FALSE, width = 200, warn = 1)
set.seed(20261012)

## 0. Settings
Everything that defines the comparison is here. The two runs differ in their ICESat-2 training years; check the **confounds** listed in `RUNS` before quoting results.

In [ ]:
## [config]
TARGET_YEAR <- 2020      # the only year with both a single-year and a multi-year v3.1 AGBD map
BASE <- "s3://maap-ops-workspace/shared/montesano/DPS_tile_lists/BOREAL_MAP"

RUNS <- list(
    single = list(label  = "Single-year (2020 ICESat-2)",
                  tindex = file.path(BASE, "v3.1.0/AGB_H30_2020/full_run/AGB_tindex_master.csv")),
    multi  = list(label  = "Multi-year (2019-2024 ICESat-2)",
                  tindex = file.path(BASE, "v3.1.0/AGB_H30_2020/full_run_niter30_multiyear_atl08v6/AGB_tindex_master.csv"))
)
# Known differences besides the training years (state these alongside any result):
CONFOUNDS <- c("multi-year run uses niter = 30 (affects the uncertainty band, sd)",
               "runs were made at different dates (Aug vs Sep 2025); check both used the same mapBoreal commit")

# Tiles: AEA grid of 90 km tiles (the gpkg has no CRS; this is the map CRS)
BOREAL_TILES_FN <- "/projects/shared-buckets/montesano/databank/boreal_tiles_v004_model_ready.gpkg"
AEA <- st_crs("+proj=aea +lat_0=40 +lon_0=180 +lat_1=50 +lat_2=70 +x_0=0 +y_0=0 +ellps=GRS80 +towgs84=0,0,0,0,0,0,0 +units=m +no_defs")

# §1 tile sample: stratified by continent x latitude band, plus tiles examined in the earlier notebook
LAT_BREAKS          <- seq(45, 75, by = 5)
N_TILES_PER_STRATUM <- 4
EXTRA_TILES         <- c(1209, 1775, 1822, 1841, 1902, 2832, 3173, 3456, 37522)
N_PIX_PER_TILE      <- 2000           # §3 paired pixel sample per tile

# §2 training-data metrics
PEAK_SEASON_DOY <- c(182, 243)        # Jul 1 - Aug 31
COVER_CELL_M    <- 3000               # "share of tile covered" = share of 3 km cells with >= 1 training obs

# §4 field plots: per-plot screening table written by maap_tools/notebooks/validation_v3.1_height_agbd.ipynb
VAL_DIR       <- "/projects/my-private-bucket/validation_v3.1"
VAL_PLOTS_FN  <- Sys.getenv("CMP_VAL_PLOTS_FN")   # default: the latest validation run
if (VAL_PLOTS_FN == "") VAL_PLOTS_FN <- tail(c(NA, sort(Sys.glob(file.path(VAL_DIR, "*", "validation_plots_v3.1_screened.gpkg")))), 1)
NONFOREST_AGBD <- 5                   # Mg/ha; same threshold as the validation notebook
MATURE_AGE     <- 60                  # yr; §5 uses mature, undisturbed stands

# §5 temporal stability. v3.1 has a single-year AGBD map for 2020 only, so the default compares HEIGHT series:
# single-year = v3.0 (2019-2024), multi-year = v3.1. That mixes model versions; it is labelled as such.
RUN_STABILITY <- TRUE
STABILITY <- list(
    variable  = "Vegetation height [m]",
    years     = 2019:2024,
    tindex    = list(single = function(y) file.path(BASE, sprintf("v3.0.0/Ht_H30_%d/full_run/HT_tindex_master.csv", y)),
                     multi  = function(y) file.path(BASE, sprintf("v3.1.0/Ht_H30_%d/full_run_niter30_multiyear_atl08v6/HT_tindex_master.csv", y))),
    caveat    = "single-year series is v3.0, multi-year series is v3.1 (version differs as well as training years)",
    max_tiles = 80
)

N_BOOT        <- 1000
N_CORES       <- 1                    # >1 uses parallel::mclapply over tiles
OUT_DIR       <- file.path("/projects/my-private-bucket/model_strategy_comparison", format(Sys.Date(), "%Y%m%d"))
CACHE_DIR     <- "/projects/my-private-bucket/model_strategy_comparison/cache"   # per-tile results; reused across days
REFRESH_CACHE <- FALSE
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(CACHE_DIR, recursive = TRUE, showWarnings = FALSE)

## Data access
All S3 reads go through these five functions (tile index, run folder listing, candidate obs, training obs, map COG).

In [ ]:
## [data-access]
suppressPackageStartupMessages({
    library(aws.s3)
    library(arrow)
    library(sfarrow)
})
Sys.setenv(AWS_DEFAULT_REGION = "us-west-2")

# On MAAP, aws.s3 no longer finds the workspace's AWS credentials by itself (403 Access Denied), but the AWS CLI does.
# Copy the CLI's current credentials into this R session's environment, where aws.s3, arrow and GDAL all look for them.
# They are temporary, so they are refreshed every `max_age_min` minutes.
.s3_auth_time <- NULL
s3_auth <- function(max_age_min = 30) {
    if (!is.null(.s3_auth_time) && difftime(Sys.time(), .s3_auth_time, units = "mins") < max_age_min) return(invisible())
    out <- tryCatch(suppressWarnings(system2("aws", c("configure", "export-credentials", "--format", "env-no-export"),
                                             stdout = TRUE, stderr = FALSE)),
                    error = function(e) character(0))
    kv <- regmatches(out, regexec("^(AWS_[A-Z_]+)=(.*)$", out))
    kv <- kv[lengths(kv) == 3]
    if (!length(kv)) stop("Could not get AWS credentials from the AWS CLI. In a terminal, check that ",
                          "`aws configure export-credentials --format env-no-export` works (AWS CLI v2.9 or later).")
    do.call(Sys.setenv, setNames(lapply(kv, `[`, 3), vapply(kv, `[`, "", 2)))
    .s3_auth_time <<- Sys.time()
    invisible()
}
s3_auth()

to_vsis3 <- function(p) sub("^s3://", "/vsis3/", p)

# One row per tile; when a tile was re-run, keep the latest output (run timestamps are in the path)
read_tindex <- function(fn) {
    s3_auth()
    aws.s3::s3read_using(read.csv, object = fn) %>%
        select(tile_num, s3_path) %>%
        arrange(tile_num, desc(s3_path)) %>%
        distinct(tile_num, .keep_all = TRUE)
}

# All files in the DPS output folder of a tile's map COG
list_run_dir <- function(cog_path) {
    s3_auth()
    dir    <- dirname(sub("^s3://", "", cog_path))
    bucket <- sub("/.*", "", dir)
    prefix <- paste0(sub("^[^/]+/", "", dir), "/")
    keys   <- vapply(aws.s3::get_bucket(bucket, prefix = prefix, max = Inf), function(x) x$Key, "")
    paste0("s3://", bucket, "/", keys)
}

# Filtered ICESat-2 obs from the 3x3 tile window (merge_neighbors geoparquet; stored without a CRS)
read_candidates <- function(fn) {
    s3_auth()
    x <- sfarrow::st_read_parquet(fn)
    st_crs(x) <- AEA
    x %>% mutate(RH_98 = h_canopy) %>% select(any_of(c("RH_98", "y", "doy")))
}

# Obs actually used to fit the tile's rF model
read_training <- function(fn) {
    s3_auth()
    arrow::read_parquet(fn) %>%
        as.data.frame() %>%
        select(any_of(c("lon", "lat", "RH_98", "AGB", "y", "doy"))) %>%
        st_as_sf(coords = c("lon", "lat"), crs = 4326) %>%
        st_transform(AEA)
}

read_map <- function(cog_path) {
    s3_auth()
    terra::rast(to_vsis3(cog_path))
}

In [ ]:
STRAT_LEVELS <- c("single", "multi")
STRAT_LABELS <- c(single = RUNS$single$label, multi = RUNS$multi$label)
STRAT_COLORS <- c(single = "#d95f02", multi = "#1b9e77")
WINNER_COLORS <- c(single = STRAT_COLORS[["single"]], multi = STRAT_COLORS[["multi"]],
                   "no clear difference" = "grey70", "not tested" = "grey92", descriptive = "white")
scale_strategy <- function(...) scale_colour_manual(values = STRAT_COLORS, labels = STRAT_LABELS, name = NULL,
                                                    aesthetics = c("colour", "fill"), ...)
theme_set(theme_bw(base_size = 11) +
          theme(legend.position = "bottom", panel.grid.minor = element_blank(),
                plot.title = element_text(face = "bold"), plot.subtitle = element_text(colour = "grey30")))

# Per-tile results are cached so re-runs only fetch what is new
cached <- function(name, expr) {
    fn <- file.path(CACHE_DIR, paste0(name, ".rds"))
    if (!REFRESH_CACHE && file.exists(fn)) return(readRDS(fn))
    val <- expr
    saveRDS(val, fn)
    val
}

# Apply f to each tile; a failing tile is reported and skipped
map_tiles <- function(tiles, f, label) {
    g <- function(t) tryCatch(f(t), error = function(e) {
        message(sprintf("  tile %s skipped (%s): %s", t, label, conditionMessage(e))); NULL })
    out <- if (N_CORES > 1) parallel::mclapply(tiles, g, mc.cores = N_CORES) else lapply(tiles, g)
    message(sprintf("%s: %d of %d tiles done", label, sum(!vapply(out, is.null, TRUE)), length(tiles)))
    out
}

cog_of <- function(tile, strategy) both[[paste0("s3_path_", strategy)]][both$tile_num == tile]

extract_at_points <- function(cog_path, pts) {
    r <- read_map(cog_path)
    v <- terra::extract(r, terra::vect(st_transform(pts, st_crs(terra::crs(r)))))
    unname(as.matrix(v[, -1, drop = FALSE]))       # drop the ID column; band 1 = mean, band 2 = sd
}

rmsd <- function(p, r) sqrt(mean((p - r)^2, na.rm = TRUE))

# Cluster bootstrap: resample whole clusters (tiles, or plot groups) with replacement
cluster_boot <- function(d, stat_fn, cluster_col = "cluster", n_boot = N_BOOT, min_clusters = 5) {
    est    <- stat_fn(d)
    groups <- split(seq_len(nrow(d)), d[[cluster_col]])
    res <- tibble(metric = names(est), estimate = unname(est), lo = NA_real_, hi = NA_real_,
                  n = nrow(d), n_clusters = length(groups))
    if (length(groups) < min_clusters) return(res)
    boots <- replicate(n_boot, stat_fn(d[unlist(groups[sample.int(length(groups), replace = TRUE)], use.names = FALSE), , drop = FALSE]))
    boots <- matrix(boots, nrow = length(est))
    res %>% mutate(lo = apply(boots, 1, quantile, 0.025, na.rm = TRUE),
                   hi = apply(boots, 1, quantile, 0.975, na.rm = TRUE))
}

# Winner of a (multi - single) difference, given which direction is better
call_winner <- function(lo, hi, better) {
    mapply(function(lo, hi, better) {
        if (is.na(better)) return("descriptive")
        if (is.na(lo) || is.na(hi)) return("not tested")
        if (lo > 0) return(if (better == "higher") "multi" else "single")
        if (hi < 0) return(if (better == "higher") "single" else "multi")
        "no clear difference"
    }, lo, hi, better, USE.NAMES = FALSE)
}

fmt_ci <- function(est, lo, hi, digits = 2) {
    ifelse(is.na(lo), formatC(est, format = "f", digits = digits),
           sprintf("%s [%s, %s]", formatC(est, format = "f", digits = digits),
                   formatC(lo, format = "f", digits = digits), formatC(hi, format = "f", digits = digits)))
}
winner_phrase <- function(w) dplyr::case_when(w == "multi" ~ "multi-year better", w == "single" ~ "single-year better",
                                              TRUE ~ w)
takeaway <- function(...) cat("\nTAKEAWAY:", ..., "\n")

## 1. Tiles compared
Only tiles mapped by **both** runs are used. A stratified sample (continent × 5° latitude band) keeps high-latitude tiles, where the single-year set is thinnest, from being swamped by the many southern tiles. The tiles inspected in the earlier notebook are added.

In [ ]:
TINDEX <- map(RUNS, ~ read_tindex(.x$tindex))
both <- inner_join(TINDEX$single, TINDEX$multi, by = "tile_num", suffix = c("_single", "_multi"))

boreal_tiles <- st_read(BOREAL_TILES_FN, quiet = TRUE)
st_crs(boreal_tiles) <- AEA
cen <- st_coordinates(st_transform(st_centroid(st_geometry(boreal_tiles)), 4326))
boreal_tiles <- boreal_tiles %>%
    mutate(lon = cen[, 1], lat = cen[, 2],
           continent = ifelse(lon > -170 & lon < -30, "North America", "Eurasia"),
           lat_band  = cut(lat, LAT_BREAKS, right = FALSE,
                           labels = paste0(head(LAT_BREAKS, -1), "-", LAT_BREAKS[-1], "°N")))

tile_info <- boreal_tiles %>% st_drop_geometry() %>% select(tile_num, lon, lat, continent, lat_band)
common <- tile_info %>% filter(tile_num %in% both$tile_num)

SAMPLE_TILES <- common %>%
    filter(!is.na(lat_band)) %>%
    group_by(continent, lat_band) %>%
    slice_sample(n = N_TILES_PER_STRATUM) %>%
    ungroup() %>%
    pull(tile_num) %>%
    c(intersect(EXTRA_TILES, common$tile_num)) %>%
    unique() %>% sort()

cat(sprintf("Tiles mapped by both runs: %d | sampled for §2-3: %d\n", nrow(common), length(SAMPLE_TILES)))
print(common %>% filter(tile_num %in% SAMPLE_TILES) %>% count(continent, lat_band) %>% pivot_wider(names_from = continent, values_from = n))

options(repr.plot.width = 10, repr.plot.height = 5, repr.plot.res = 150)
ggplot() +
    geom_sf(data = boreal_tiles %>% filter(tile_num %in% common$tile_num), fill = "grey85", colour = NA) +
    geom_sf(data = boreal_tiles %>% filter(tile_num %in% SAMPLE_TILES), fill = "black", colour = NA) +
    labs(title = "Tiles compared", subtitle = sprintf("grey: mapped by both runs (%d); black: sampled for training and map comparison (%d)",
                                                     nrow(common), length(SAMPLE_TILES)))

## 2. Evidence A: what each model learns from
For each sampled tile and strategy, from the `*_train.parquet` (obs used to fit the rF model) and the `*merge_neighbors*.parquet` (all filtered candidate obs in the 3×3 tile window):

| Metric | Why it matters | Better |
|---|---|---|
| Training obs in tile | More obs → more stable rF fits and lower prediction variance | higher |
| Share of 3 km cells with ≥ 1 training obs | Even spatial coverage → fewer areas predicted far from any training data | higher |
| Median distance to nearest training obs | Same, as a distance (computed on a 1 km lattice over the tile) | lower |
| % obs in Jul–Aug, 95th pct training height | Timing and height range of the training set | descriptive |

Training obs carry no acquisition date, so each is matched to its candidate obs (nearest within 15 m and within 0.5 m in RH98). `pct_dated` reports how many matched.

In [ ]:
attach_obs_dates <- function(train, cand, max_dist = 15, max_dh = 0.5) {
    if (all(c("y", "doy") %in% names(train)) && any(!is.na(train$doy))) return(mutate(train, dated = !is.na(doy)))
    train <- train %>% mutate(y = NA_real_, doy = NA_real_, dated = FALSE)
    if (is.null(cand) || nrow(cand) == 0) return(train)
    i  <- st_nearest_feature(train, cand)
    d  <- as.numeric(st_distance(train, cand[i, ], by_element = TRUE))
    ok <- d <= max_dist & abs(train$RH_98 - cand$RH_98[i]) <= max_dh
    train$y[ok]   <- cand$y[i][ok]
    train$doy[ok] <- cand$doy[i][ok]
    train$dated   <- ok
    train
}

coverage_metrics <- function(pts, focal, cell = COVER_CELL_M, lattice = 1000) {
    grid <- st_make_grid(focal, cellsize = cell)
    grid <- grid[lengths(st_intersects(grid, focal)) > 0]
    lat_pts <- st_make_grid(focal, cellsize = lattice, what = "centers")
    d <- as.numeric(st_distance(lat_pts, pts[st_nearest_feature(lat_pts, pts), ], by_element = TRUE)) / 1000
    c(frac_cells = mean(lengths(st_intersects(grid, pts)) > 0),
      med_dist_km = median(d), p90_dist_km = unname(quantile(d, 0.9)))
}

summarise_training_tile <- function(t) {
    focal <- st_geometry(boreal_tiles[boreal_tiles$tile_num == t, ])
    res <- lapply(STRAT_LEVELS, function(s) {
        files   <- list_run_dir(cog_of(t, s))
        f_train <- grep("_train\\.parquet$", files, value = TRUE)[1]
        f_cand  <- grep("merge_neighbors.*\\.parquet$", files, value = TRUE)[1]
        if (is.na(f_train)) stop("no *_train.parquet for ", s)
        cand  <- if (!is.na(f_cand)) read_candidates(f_cand) else NULL
        train <- attach_obs_dates(read_training(f_train), cand)
        cov   <- coverage_metrics(train, focal)
        in_tile <- lengths(st_intersects(train, focal)) > 0
        tr <- st_drop_geometry(train) %>% mutate(in_tile = in_tile)
        dated <- tr %>% filter(dated)
        list(
            summary = tibble(
                tile_num = t, strategy = s,
                n_train = nrow(tr), n_train_tile = sum(in_tile),
                n_candidates = if (is.null(cand)) NA_integer_ else nrow(cand),
                frac_cells = cov[["frac_cells"]], med_dist_km = cov[["med_dist_km"]], p90_dist_km = cov[["p90_dist_km"]],
                pct_dated = 100 * mean(tr$dated),
                n_years = n_distinct(dated$y),
                pct_target_year = 100 * mean(dated$y == TARGET_YEAR),
                pct_peak_season = 100 * mean(dated$doy >= PEAK_SEASON_DOY[1] & dated$doy <= PEAK_SEASON_DOY[2]),
                rh98_p50 = median(tr$RH_98), rh98_p95 = unname(quantile(tr$RH_98, 0.95)),
                agb_p95 = if ("AGB" %in% names(tr)) unname(quantile(tr$AGB, 0.95, na.rm = TRUE)) else NA_real_),
            obs = tr %>% filter(in_tile) %>% slice_sample(n = 3000) %>%
                transmute(tile_num = t, strategy = s, RH_98, y, doy))
    })
    list(summary = bind_rows(map(res, "summary")), obs = bind_rows(map(res, "obs")))
}

train_res <- map_tiles(SAMPLE_TILES, function(t) cached(paste0("train_", t), summarise_training_tile(t)), "training data")
train_sum <- bind_rows(map(compact(train_res), "summary")) %>% left_join(tile_info, by = "tile_num")
train_obs <- bind_rows(map(compact(train_res), "obs"))
train_sum %>% group_by(strategy) %>%
    summarise(tiles = n(), across(c(n_train_tile, frac_cells, med_dist_km, pct_dated, pct_peak_season, rh98_p95), ~ round(median(.x, na.rm = TRUE), 2)))

In [ ]:
TRAIN_METRICS <- tribble(
    ~metric,           ~label,                                         ~better,  ~log2ratio,
    "n_train_tile",    "Training obs in tile",                         "higher", TRUE,
    "frac_cells",      "Share of 3-km cells with >= 1 training obs",   "higher", FALSE,
    "med_dist_km",     "Median distance to nearest training obs (km)", "lower",  FALSE,
    "pct_peak_season", "% training obs in Jul-Aug",                    NA,       FALSE,
    "rh98_p95",        "95th pct training height (m)",                 NA,       FALSE)

# Paired by tile: difference (multi - single), or log2 ratio for counts; bootstrap over tiles
train_paired <- train_sum %>%
    select(tile_num, strategy, all_of(TRAIN_METRICS$metric)) %>%
    pivot_longer(-c(tile_num, strategy), names_to = "metric") %>%
    pivot_wider(names_from = strategy, values_from = value) %>%
    filter(!is.na(single), !is.na(multi)) %>%
    left_join(TRAIN_METRICS, by = "metric")

train_tests <- train_paired %>%
    group_by(metric) %>%
    group_modify(function(d, key) {
        lr <- TRAIN_METRICS$log2ratio[TRAIN_METRICS$metric == key$metric]
        f  <- if (lr) function(x) c(diff = mean(log2(pmax(x$multi, 1) / pmax(x$single, 1))))
              else    function(x) c(diff = mean(x$multi - x$single))
        cluster_boot(d %>% mutate(cluster = tile_num), f) %>%
            select(-metric) %>%
            mutate(single = median(d$single), multi = median(d$multi),
                   pct_tiles_multi_higher = 100 * mean(d$multi > d$single))
    }) %>% ungroup() %>%
    left_join(TRAIN_METRICS, by = "metric") %>%
    mutate(winner = call_winner(lo, hi, better))
print(train_tests %>% select(label, single, multi, estimate, lo, hi, pct_tiles_multi_higher, winner))

# Figure A1: each tile's two values, tiles ordered by latitude
options(repr.plot.width = 14, repr.plot.height = 9, repr.plot.res = 150)
dumbbell <- train_paired %>%
    filter(!is.na(better)) %>%
    left_join(tile_info, by = "tile_num") %>%
    mutate(tile = reorder(paste0(tile_num, " (", round(lat), "°N)"), lat))
pA1 <- ggplot(dumbbell, aes(y = tile)) +
    geom_segment(aes(x = single, xend = multi, yend = tile), colour = "grey60") +
    geom_point(aes(x = single, colour = "single"), size = 1.8) +
    geom_point(aes(x = multi, colour = "multi"), size = 1.8) +
    facet_wrap(~ label, scales = "free_x", nrow = 1) +
    scale_strategy() +
    labs(x = NULL, y = "Tile (ordered by latitude)", title = "A1. Training data per tile",
         subtitle = "Each row is one tile; the line spans the two strategies")
pA1

**Reading A1.** A long line means the two training sets differ a lot for that tile. If the multi-year points sit consistently to the right in the first two panels (more obs, more cells covered) and to the left in the third (shorter distance to the nearest obs), multi-year training samples the tile more densely and more evenly. Watch the northern rows: there the single-year set loses many obs to the night-only filter during the polar summer.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 4.5, repr.plot.res = 150)
obs_dated <- train_obs %>% filter(!is.na(doy))
pA2 <- ggplot(obs_dated, aes(x = doy, fill = strategy)) +
    geom_histogram(binwidth = 7, position = "identity", alpha = 0.55) +
    annotate("rect", xmin = PEAK_SEASON_DOY[1], xmax = PEAK_SEASON_DOY[2], ymin = -Inf, ymax = Inf, alpha = 0.08) +
    scale_x_continuous(breaks = c(152, 182, 213, 244, 274), labels = c("Jun 1", "Jul 1", "Aug 1", "Sep 1", "Oct 1")) +
    scale_strategy() +
    labs(x = "Acquisition date", y = "Training obs (sampled tiles)", title = "A2. When the training obs were acquired",
         subtitle = "Shaded: Jul-Aug peak season")
pA3 <- ggplot(train_obs, aes(x = RH_98, colour = strategy)) +
    stat_ecdf(linewidth = 0.9) +
    scale_x_continuous(limits = c(0, quantile(train_obs$RH_98, 0.995, na.rm = TRUE))) +
    scale_strategy() + guides(colour = "none") +      # legend shown once, in A2
    labs(x = "Training height RH98 (m)", y = "Cumulative share", title = "A3. Height range of training obs",
         subtitle = "A curve further right reaches taller stands")
pA4 <- ggplot(obs_dated %>% filter(strategy == "multi"), aes(x = factor(y))) +
    geom_bar(fill = STRAT_COLORS[["multi"]]) +
    labs(x = "ICESat-2 year", y = "Training obs", title = "A4. Years in the multi-year pool",
         subtitle = sprintf("share from %d: %.0f%%", TARGET_YEAR, 100 * mean(obs_dated$y[obs_dated$strategy == "multi"] == TARGET_YEAR)))
print(pA2 + pA3 + pA4 + plot_layout(widths = c(2, 1.3, 1)))
tr_n <- train_tests %>% filter(metric == "n_train_tile")
tr_c <- train_tests %>% filter(metric == "frac_cells")
takeaway(sprintf("multi-year training has %.1fx the obs per tile [%.1fx, %.1fx] (more in %.0f%% of tiles) and covers %+.0f percentage points more of each tile's 3-km cells [%+.0f, %+.0f].",
                 2^tr_n$estimate, 2^tr_n$lo, 2^tr_n$hi, tr_n$pct_tiles_multi_higher,
                 100 * tr_c$estimate, 100 * tr_c$lo, 100 * tr_c$hi))

## 3. Evidence B: where the two 2020 maps disagree
The same `N_PIX_PER_TILE` pixels (a regular grid) are read from both 2020 maps in every sampled tile. This shows *where* the choice of strategy changes the map, and whether disagreement follows the training-data differences from §2. It cannot say which map is right; that is §4.

In [ ]:
compare_maps_tile <- function(t) {
    rs <- read_map(cog_of(t, "single"))
    xy <- terra::spatSample(rs[[1]], size = N_PIX_PER_TILE, method = "regular", xy = TRUE, na.rm = FALSE)[, c("x", "y")]
    pts <- st_as_sf(xy, coords = c("x", "y"), crs = st_crs(terra::crs(rs)))
    vs <- extract_at_points(cog_of(t, "single"), pts)
    vm <- extract_at_points(cog_of(t, "multi"), pts)
    tibble(tile_num = t,
           agb_single = vs[, 1], sd_single = if (ncol(vs) > 1) vs[, 2] else NA_real_,
           agb_multi  = vm[, 1], sd_multi  = if (ncol(vm) > 1) vm[, 2] else NA_real_) %>%
        filter(!is.na(agb_single), !is.na(agb_multi))
}

pix <- bind_rows(map_tiles(SAMPLE_TILES, function(t) cached(paste0("pix_", t, "_", N_PIX_PER_TILE), compare_maps_tile(t)), "map comparison")) %>%
    mutate(diff = agb_multi - agb_single) %>%
    left_join(tile_info, by = "tile_num")

tile_diff <- pix %>% group_by(tile_num, lat, lat_band, continent) %>%
    summarise(n = n(), mean_single = mean(agb_single), mean_multi = mean(agb_multi), mean_diff = mean(diff),
              rmsd_between = sqrt(mean(diff^2)), sd_single = mean(sd_single), sd_multi = mean(sd_multi)) %>%
    ungroup() %>%
    left_join(train_sum %>% select(tile_num, strategy, n_train_tile) %>%
                  pivot_wider(names_from = strategy, values_from = n_train_tile, names_prefix = "n_"), by = "tile_num") %>%
    mutate(log2_n_ratio = log2(pmax(n_multi, 1) / pmax(n_single, 1)))

map_tests <- bind_rows(
    cluster_boot(pix %>% mutate(cluster = tile_num), function(d) c(mean_diff = mean(d$diff))),
    cluster_boot(pix %>% filter(!is.na(sd_single), !is.na(sd_multi)) %>% mutate(cluster = tile_num),
                 function(d) c(sd_diff = mean(d$sd_multi - d$sd_single))))
map_tests

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, repr.plot.res = 150)
lim <- quantile(c(pix$agb_single, pix$agb_multi), 0.995)
pB1 <- ggplot(pix, aes(agb_single, agb_multi)) +
    geom_bin2d(bins = 70) +
    geom_abline(linetype = "dashed") +
    scale_fill_viridis_c(trans = "log10", name = "pixels") +
    coord_equal(xlim = c(0, lim), ylim = c(0, lim)) +
    labs(x = "Single-year AGBD (Mg/ha)", y = "Multi-year AGBD (Mg/ha)", title = "B1. Same pixels, two maps",
         subtitle = sprintf("%s pixels, %d tiles", comma(nrow(pix)), n_distinct(pix$tile_num)))
pB2 <- ggplot(pix, aes(x = lat_band, y = diff, fill = continent)) +
    geom_hline(yintercept = 0) +
    geom_boxplot(outlier.shape = NA, width = 0.7) +
    coord_cartesian(ylim = quantile(pix$diff, c(0.02, 0.98))) +
    scale_fill_manual(values = c("North America" = "grey80", "Eurasia" = "grey50"), name = NULL) +
    labs(x = NULL, y = "Multi - single (Mg/ha)", title = "B2. Disagreement by latitude",
         subtitle = "Box: pixel differences within each band (whiskers trimmed)") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))
pB3 <- ggplot(tile_diff, aes(log2_n_ratio, rmsd_between)) +
    geom_point(aes(colour = lat)) +
    geom_smooth(method = "lm", formula = y ~ x, colour = "black", linewidth = 0.6) +
    scale_colour_viridis_c(name = "Latitude") +
    labs(x = "log2(multi / single training obs in tile)", y = "RMS difference between maps (Mg/ha)",
         title = "B3. Disagreement vs training-data gap",
         subtitle = "Each point is a tile")
print(pB1 + pB2 + pB3 + plot_layout(widths = c(1, 1.3, 1.1)))

md_cor <- tryCatch(with(tile_diff, suppressWarnings(cor.test(log2_n_ratio, rmsd_between, method = "spearman", exact = FALSE))),
                   error = function(e) list(estimate = NA_real_, p.value = NA_real_))
md_diff <- map_tests %>% filter(metric == "mean_diff")
takeaway(sprintf("the multi-year map is %+.1f Mg/ha [%+.1f, %+.1f] relative to single-year on average; %.0f%% of pixels differ by > 20 Mg/ha. Tiles where multi-year has relatively more training obs disagree %s (Spearman rho = %.2f, p = %.3f).",
                 md_diff$estimate, md_diff$lo, md_diff$hi, 100 * mean(abs(pix$diff) > 20),
                 ifelse(md_cor$estimate > 0, "more", "less"), md_cor$estimate, md_cor$p.value))

**Reading B.** B1: points on the dashed line are pixels where the strategy makes no difference; a cloud above or below it is a systematic shift. B2: if the boxes move away from zero toward the north, the strategy matters most where the single-year set is thinnest. B3: a rising trend means the maps disagree most where the training sets differ most, which ties map differences to training data rather than to noise.

The two maps' own uncertainty bands (`sd_diff` above) are listed in the scorecard as descriptive only: the multi-year run used `niter = 30`, which changes the sd independently of training data.

## 4. Evidence C: accuracy against field plots (who is right)
Field plots come from the screened table written by `validation_v3.1_height_agbd.ipynb`. Kept: plots not excluded a priori for AGBD, with a stand-age class of *undisturbed* or *marginal*, or non-forest plots without a mapped age (same rule as the validation's default tier).

**Design: paired.** Both 2020 maps are read at the same plots and compared against the same reference value. Most plots were measured years before 2020, so both maps carry the same reference-age error; it cancels in the paired difference. What remains is the effect of the training strategy.

Metrics per strategy: bias (mean map − field), RMSD, MAE. Paired differences (multi − single) with 95 % CIs from a cluster bootstrap over plot groups (source × group × field year). Also the share of plots where the multi-year map is closer to the field value (50 % = no difference).

In [ ]:
if (is.na(VAL_PLOTS_FN) || !file.exists(VAL_PLOTS_FN))
    stop("No screened field plots found. Run maap_tools/notebooks/validation_v3.1_height_agbd.ipynb first, or set CMP_VAL_PLOTS_FN.")
message("Field plots: ", VAL_PLOTS_FN)

val_all <- st_read(VAL_PLOTS_FN, quiet = TRUE) %>%
    mutate(across(any_of(c("excl_agbd", "excl_ht")), as.logical), age_class = as.character(age_class))
plots_tiled <- val_all %>%
    st_transform(AEA) %>%
    st_join(boreal_tiles %>% select(tile_num), join = st_intersects, left = FALSE) %>%
    distinct(uid, .keep_all = TRUE) %>%
    filter(tile_num %in% both$tile_num)

acc_plots <- plots_tiled %>%
    filter(!excl_agbd, !is.na(ref_agbd),
           age_class %in% c("undisturbed", "marginal") | (age_class == "no mapped age" & ref_agbd < NONFOREST_AGBD))

extract_plots_tile <- function(t, pts, cogs = list(single = cog_of(t, "single"), multi = cog_of(t, "multi"))) {
    p <- pts[pts$tile_num == t, ]
    out <- tibble(uid = p$uid)
    for (s in names(cogs)) out[[paste0("pred_", s)]] <- extract_at_points(cogs[[s]], p)[, 1]
    out
}
acc_tiles <- sort(unique(acc_plots$tile_num))
pred_plots <- bind_rows(map_tiles(acc_tiles, function(t) cached(paste0("plots_", TARGET_YEAR, "_", t, "_n", sum(acc_plots$tile_num == t)), extract_plots_tile(t, acc_plots)), "field plots"))

pairs <- acc_plots %>% st_drop_geometry() %>%
    inner_join(pred_plots, by = "uid") %>%
    filter(!is.na(pred_single), !is.na(pred_multi)) %>%
    mutate(cluster = paste(source, group_name, year_obs),
           err_single = pred_single - ref_agbd, err_multi = pred_multi - ref_agbd)
cat(sprintf("Paired plots: %d in %d plot groups, %d tiles (%s)\n", nrow(pairs), n_distinct(pairs$cluster),
            n_distinct(pairs$tile_num), paste(names(table(pairs$region)), table(pairs$region), sep = ": ", collapse = ", ")))

In [ ]:
per_strategy <- function(d) c(bias_single = mean(d$err_single), bias_multi = mean(d$err_multi),
                              rmsd_single = rmsd(d$pred_single, d$ref_agbd), rmsd_multi = rmsd(d$pred_multi, d$ref_agbd),
                              mae_single = mean(abs(d$err_single)), mae_multi = mean(abs(d$err_multi)))
paired_stats <- function(d) c(d_rmsd    = rmsd(d$pred_multi, d$ref_agbd) - rmsd(d$pred_single, d$ref_agbd),
                              d_mae     = mean(abs(d$err_multi)) - mean(abs(d$err_single)),
                              d_absbias = abs(mean(d$err_multi)) - abs(mean(d$err_single)),
                              pct_multi_closer_minus50 = 100 * mean(abs(d$err_multi) < abs(d$err_single)) - 50)

HEADLINE <- "All boreal plots"
boreal_pairs <- pairs %>% filter(region != "Temperate")
acc_tests <- bind_rows(
    cluster_boot(boreal_pairs, paired_stats) %>% mutate(domain = HEADLINE),
    boreal_pairs %>% group_by(domain = region) %>% group_modify(~ cluster_boot(.x, paired_stats)) %>% ungroup(),
    if (any(pairs$region == "Temperate"))
        cluster_boot(pairs %>% filter(region == "Temperate"), paired_stats) %>% mutate(domain = "Temperate (not in headline)")) %>%
    mutate(better = ifelse(metric == "pct_multi_closer_minus50", "higher", "lower"),
           winner = call_winner(lo, hi, better))
acc_abs <- cluster_boot(boreal_pairs, per_strategy)
print(acc_abs)
acc_tests %>% filter(domain == HEADLINE) %>% select(metric, estimate, lo, hi, n, n_clusters, winner)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5.5, repr.plot.res = 150)
long_pairs <- boreal_pairs %>%
    select(uid, region, ref_agbd, single = pred_single, multi = pred_multi) %>%
    pivot_longer(c(single, multi), names_to = "strategy", values_to = "pred") %>%
    mutate(strategy = factor(strategy, STRAT_LEVELS))
lab <- long_pairs %>% group_by(strategy) %>%
    summarise(txt = sprintf("bias %+.1f | RMSD %.1f | MAE %.1f Mg/ha", mean(pred - ref_agbd), rmsd(pred, ref_agbd), mean(abs(pred - ref_agbd))))
lim <- quantile(c(long_pairs$pred, long_pairs$ref_agbd), 0.995, na.rm = TRUE)
pC1 <- ggplot(long_pairs, aes(ref_agbd, pred, colour = strategy)) +
    geom_abline(linetype = "dashed") +
    geom_point(alpha = 0.35, size = 0.9) +
    geom_text(data = lab, aes(x = 0, y = lim, label = txt), hjust = 0, vjust = 1, size = 3.2, colour = "black", inherit.aes = FALSE) +
    facet_wrap(~ strategy, labeller = as_labeller(STRAT_LABELS)) +
    coord_equal(xlim = c(0, lim), ylim = c(0, lim)) +
    scale_strategy() + guides(colour = "none") +
    labs(x = "Field AGBD (Mg/ha)", y = sprintf("Map AGBD %d (Mg/ha)", TARGET_YEAR), title = "C1. Both maps against the same field plots")

forest <- acc_tests %>% filter(metric %in% c("d_rmsd", "d_absbias", "d_mae")) %>%
    mutate(metric = recode(metric, d_rmsd = "RMSD", d_mae = "MAE", d_absbias = "|bias|"),
           domain = factor(domain, rev(unique(domain))))
pC2 <- ggplot(forest, aes(estimate, domain, colour = winner)) +
    geom_vline(xintercept = 0) +
    geom_errorbar(aes(xmin = lo, xmax = hi), width = 0.25, orientation = "y") +
    geom_point(size = 2.3) +
    facet_wrap(~ metric, scales = "free_x") +
    scale_colour_manual(values = WINNER_COLORS, name = NULL) +
    labs(x = "Multi - single (Mg/ha); left of 0 = multi-year more accurate", y = NULL,
         title = "C2. Paired accuracy difference", subtitle = "95% cluster-bootstrap CI; n plots per row in the table above")
print(pC1 + pC2 + plot_layout(widths = c(1.4, 1)))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4.5, repr.plot.res = 150)
by_ref <- boreal_pairs %>%
    mutate(ref_bin = cut(ref_agbd, c(-Inf, 5, 25, 50, 100, 150, Inf), labels = c("<5", "5-25", "25-50", "50-100", "100-150", ">150"))) %>%
    group_by(ref_bin) %>%
    summarise(n = n(), single = mean(abs(err_single)), multi = mean(abs(err_multi)),
              pct_multi_closer = 100 * mean(abs(err_multi) < abs(err_single))) %>%
    pivot_longer(c(single, multi), names_to = "strategy", values_to = "mae")
pC3 <- ggplot(by_ref, aes(ref_bin, mae, colour = strategy, group = strategy)) +
    geom_line(linewidth = 0.9) + geom_point(size = 2) +
    geom_text(data = by_ref %>% filter(strategy == "multi"), aes(y = 0, label = paste0("n=", n)), colour = "grey40", vjust = 0, size = 3) +
    scale_strategy() +
    labs(x = "Field AGBD (Mg/ha)", y = "Mean absolute error (Mg/ha)", title = "C3. Where the accuracy difference comes from",
         subtitle = "Lower is better")
print(pC3)

acc_main <- acc_tests %>% filter(domain == HEADLINE)
d_r <- acc_main %>% filter(metric == "d_rmsd")
d_c <- acc_main %>% filter(metric == "pct_multi_closer_minus50")
takeaway(sprintf("against %d boreal field plots (%d groups), RMSD single = %.1f, multi = %.1f Mg/ha; difference %s Mg/ha (%s). The multi-year map is closer at %.0f%% of plots [%.0f, %.0f].",
                 d_r$n, d_r$n_clusters, acc_abs$estimate[acc_abs$metric == "rmsd_single"], acc_abs$estimate[acc_abs$metric == "rmsd_multi"],
                 fmt_ci(d_r$estimate, d_r$lo, d_r$hi, 1), winner_phrase(d_r$winner), 50 + d_c$estimate, 50 + d_c$lo, 50 + d_c$hi))

**Reading C.** C1 shows both maps against the field values: scatter along the dashed line is good. C2 is the decision figure: a point left of zero with its whole interval left of zero means the multi-year map is more accurate in that domain; an interval crossing zero means the plots cannot tell the two apart there. C3 shows where along the biomass range the difference arises. Single-year models often fall short in tall, high-biomass stands when the target year has few such obs.

Field plots were mostly measured before 2016, the reference is plot-scale (314–707 m²) against 900 m² pixels, and plots are clustered in groups. These affect both maps equally in the paired design, but they limit how small a difference can be detected.

## 5. Evidence D: temporal stability
In mature (age ≥ `MATURE_AGE`), undisturbed stands, real AGBD and height change little from year to year. Year-to-year variation in the map there is mostly noise, so the strategy with **less** variation is more trustworthy for trends.

Per plot: the standard deviation across years and the mean absolute year-to-year change. Paired by plot (multi − single), bootstrapped over tiles.

**Two caveats, read before quoting.**
- v3.1 has only one single-year map (2020), so the default compares *height* series: single-year v3.0 against multi-year v3.1. Version differs as well as training, so this section is supporting evidence only.
- Multi-year models share their training pool across years, so they are smoother partly by construction. Lower noise is only a virtue if real change is still detected. The LVIS 2017→2019 change test in `validation_v3.1_lidar_LVIS_GLiHT.ipynb` checks that.

In [ ]:
stab_tests <- NULL
if (RUN_STABILITY) {
    STAB_TINDEX <- map(STABILITY$tindex, function(f) {
        bind_rows(lapply(STABILITY$years, function(y) tryCatch(read_tindex(f(y)) %>% mutate(year = y),
                                                               error = function(e) { message("  no tindex for ", y, ": ", conditionMessage(e)); NULL })))
    })
    stab_plots <- plots_tiled %>%
        filter(!excl_ht, age_class == "undisturbed", !is.na(age_map), age_map >= MATURE_AGE)
    if (any(vapply(STAB_TINDEX, nrow, 1L) == 0)) stop("no stability tile index found for one strategy; set RUN_STABILITY <- FALSE or fix STABILITY$tindex")
    stab_tiles <- intersect(unique(stab_plots$tile_num), intersect(STAB_TINDEX$single$tile_num, STAB_TINDEX$multi$tile_num))
    if (length(stab_tiles) > STABILITY$max_tiles) stab_tiles <- sort(sample(stab_tiles, STABILITY$max_tiles))

    extract_series_tile <- function(t) {
        p <- stab_plots[stab_plots$tile_num == t, ]
        bind_rows(lapply(STRAT_LEVELS, function(s) {
            ti <- STAB_TINDEX[[s]] %>% filter(tile_num == t)
            bind_rows(lapply(seq_len(nrow(ti)), function(i)
                tibble(uid = p$uid, tile_num = t, strategy = s, year = ti$year[i],
                       value = extract_at_points(ti$s3_path[i], p)[, 1])))
        }))
    }
    series <- bind_rows(map_tiles(stab_tiles, function(t) cached(paste0("series_", t, "_n", sum(stab_plots$tile_num == t)), extract_series_tile(t)), "time series")) %>%
        filter(!is.na(value))

    per_plot <- series %>%
        arrange(uid, strategy, year) %>%
        group_by(uid, tile_num, strategy) %>%
        summarise(n_years = n(), sd = sd(value), mean_abs_step = mean(abs(diff(value))), mean = mean(value)) %>%
        ungroup() %>%
        filter(n_years >= 4)
    stab_pairs <- per_plot %>%
        select(uid, tile_num, strategy, sd, mean_abs_step) %>%
        pivot_wider(names_from = strategy, values_from = c(sd, mean_abs_step)) %>%
        filter(!is.na(sd_single), !is.na(sd_multi))
    stab_tests <- cluster_boot(stab_pairs %>% mutate(cluster = tile_num),
                               function(d) c(d_sd = median(d$sd_multi - d$sd_single),
                                             d_step = median(d$mean_abs_step_multi - d$mean_abs_step_single))) %>%
        mutate(better = "lower", winner = call_winner(lo, hi, better),
               single = c(median(stab_pairs$sd_single), median(stab_pairs$mean_abs_step_single)),
               multi  = c(median(stab_pairs$sd_multi),  median(stab_pairs$mean_abs_step_multi)))
    cat(sprintf("Mature undisturbed plots with >= 4 years in both series: %d (%d tiles)\n", nrow(stab_pairs), n_distinct(stab_pairs$tile_num)))
    print(stab_tests)
} else message("RUN_STABILITY = FALSE: section skipped")

In [ ]:
if (!is.null(stab_tests)) {
    options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 150)
    show_uid <- sample(unique(stab_pairs$uid), min(40, n_distinct(stab_pairs$uid)))
    traj <- series %>% filter(uid %in% show_uid) %>%
        group_by(uid, strategy) %>% mutate(anomaly = value - mean(value)) %>% ungroup() %>%
        mutate(strategy = factor(strategy, STRAT_LEVELS))
    pD1 <- ggplot(traj, aes(year, anomaly, group = uid, colour = strategy)) +
        geom_hline(yintercept = 0) +
        geom_line(alpha = 0.5) +
        facet_wrap(~ strategy, labeller = as_labeller(STRAT_LABELS)) +
        scale_strategy() + guides(colour = "none") +
        labs(x = NULL, y = paste("Departure from plot mean,", STABILITY$variable), title = "D1. Year-to-year variation in mature, undisturbed stands",
             subtitle = paste(length(show_uid), "random plots; flatter lines = less noise. Caveat:", STABILITY$caveat))
    pD2 <- ggplot(per_plot %>% mutate(strategy = factor(strategy, STRAT_LEVELS)), aes(strategy, sd, fill = strategy)) +
        geom_boxplot(outlier.shape = NA, width = 0.6) +
        coord_cartesian(ylim = c(0, quantile(per_plot$sd, 0.97))) +
        scale_x_discrete(labels = STRAT_LABELS) + scale_strategy() + guides(fill = "none") +
        labs(x = NULL, y = paste("SD across years,", STABILITY$variable), title = "D2. Per-plot noise")
    print(pD1 + pD2 + plot_layout(widths = c(2.2, 1)))
    s <- stab_tests %>% filter(metric == "d_sd")
    takeaway(sprintf("in %d mature undisturbed plots, the median year-to-year SD is %.2f (single) vs %.2f (multi); paired median difference %s (%s). %s.",
                     nrow(stab_pairs), s$single, s$multi, fmt_ci(s$estimate, s$lo, s$hi), winner_phrase(s$winner), STABILITY$caveat))
}

## 6. Scorecard and verdict
One row per criterion: the value for each strategy, the paired difference (multi − single) with its 95 % CI, and which strategy wins. A win needs the CI to exclude zero. **Role** sets how much a row counts: field-plot accuracy (§4) decides; training data (§2) and stability (§5) support or qualify; map agreement (§3) is descriptive.

In [ ]:
score <- bind_rows(
    acc_tests %>% filter(domain == HEADLINE, metric %in% c("d_rmsd", "d_absbias", "pct_multi_closer_minus50")) %>%
        transmute(evidence = "C. Accuracy vs field plots", role = "primary",
                  criterion = recode(metric, d_rmsd = "RMSD vs field plots (Mg/ha)", d_absbias = "|Bias| vs field plots (Mg/ha)",
                                     pct_multi_closer_minus50 = "Plots where multi-year is closer (% - 50)"),
                  single = c(d_rmsd = acc_abs$estimate[acc_abs$metric == "rmsd_single"],
                             d_absbias = abs(acc_abs$estimate[acc_abs$metric == "bias_single"]),
                             pct_multi_closer_minus50 = NA)[metric],
                  multi  = c(d_rmsd = acc_abs$estimate[acc_abs$metric == "rmsd_multi"],
                             d_absbias = abs(acc_abs$estimate[acc_abs$metric == "bias_multi"]),
                             pct_multi_closer_minus50 = NA)[metric],
                  estimate, lo, hi, better, winner),
    train_tests %>% transmute(evidence = "A. Training data", role = ifelse(is.na(better), "descriptive", "supporting"),
                              criterion = ifelse(log2ratio, paste(label, "(log2 ratio)"), label),
                              single, multi, estimate, lo, hi, better, winner),
    map_tests %>% transmute(evidence = "B. Map agreement", role = "descriptive",
                            criterion = recode(metric, mean_diff = "Mean AGBD difference (Mg/ha)", sd_diff = "Mean map uncertainty sd (Mg/ha; niter differs)"),
                            single = NA_real_, multi = NA_real_, estimate, lo, hi, better = NA_character_, winner = "descriptive"),
    if (!is.null(stab_tests)) stab_tests %>%
        transmute(evidence = "D. Temporal stability", role = "supporting",
                  criterion = recode(metric, d_sd = paste("Median year-to-year SD,", STABILITY$variable),
                                     d_step = paste("Median abs. annual step,", STABILITY$variable)),
                  single, multi, estimate, lo, hi, better, winner)
) %>%
    mutate(difference = fmt_ci(estimate, lo, hi),
           role = factor(role, c("primary", "supporting", "descriptive")))

score %>% arrange(role, evidence) %>%
    transmute(role, evidence, criterion, single = round(single, 2), multi = round(multi, 2),
              `multi - single [95% CI]` = difference, `better if` = better, winner)

In [ ]:
wins <- score %>% filter(role != "descriptive") %>% count(role, winner)
prim <- score %>% filter(role == "primary", criterion == "RMSD vs field plots (Mg/ha)")
supp <- score %>% filter(role == "supporting", winner %in% STRAT_LEVELS) %>% count(winner)
supp_txt <- if (nrow(supp)) paste(sprintf("%d supporting criteria favour %s-year", supp$n, supp$winner), collapse = "; ") else "no supporting criterion shows a clear difference"

verdict <- if (prim$winner %in% STRAT_LEVELS) {
    sprintf("VERDICT: the %s-year model is better. It is more accurate against independent field plots (RMSD %.1f vs %.1f Mg/ha; difference %s). %s.",
            prim$winner, prim[[prim$winner]], prim[[setdiff(STRAT_LEVELS, prim$winner)]], prim$difference, supp_txt)
} else if (nrow(supp) == 1) {
    sprintf("VERDICT: field plots cannot separate the two maps (RMSD difference %s Mg/ha). The case for the %s-year model rests on supporting evidence: %s.",
            prim$difference, supp$winner, supp_txt)
} else {
    sprintf("VERDICT: inconclusive. Field plots cannot separate the maps (RMSD difference %s Mg/ha), and %s.", prim$difference, supp_txt)
}
cat(strwrap(verdict, 110), sep = "\n")
cat("\nConfounds to state with this result:", paste0("\n - ", c(CONFOUNDS, if (!is.null(stab_tests)) paste("stability:", STABILITY$caveat))), "\n")

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 0.45 * nrow(score) + 2, repr.plot.res = 150)
card <- score %>%
    mutate(row = factor(paste0(evidence, ": ", criterion), rev(paste0(evidence, ": ", criterion))),
           winner_lab = ifelse(winner %in% STRAT_LEVELS, paste0(winner, "-year"), winner))
pS <- ggplot(card, aes(y = row)) +
    geom_tile(aes(x = 1, fill = winner), colour = "white", width = 0.95) +
    geom_text(aes(x = 1, label = winner_lab), size = 3.3) +
    geom_text(aes(x = 2, label = ifelse(is.na(single), "", formatC(single, format = "f", digits = 2))), size = 3.3) +
    geom_text(aes(x = 3, label = ifelse(is.na(multi), "", formatC(multi, format = "f", digits = 2))), size = 3.3) +
    geom_text(aes(x = 4.6, label = difference), size = 3.3) +
    geom_text(aes(x = 5.9, label = as.character(role)), size = 3.3, colour = "grey35") +
    scale_x_continuous(breaks = c(1, 2, 3, 4.6, 5.9), labels = c("Winner", "Single", "Multi", "Multi - single [95% CI]", "Role"),
                       position = "top", limits = c(0.5, 6.5)) +
    scale_fill_manual(values = WINNER_COLORS, guide = "none") +
    labs(x = NULL, y = NULL, title = "Scorecard: single-year vs multi-year ICESat-2 training",
         caption = paste(strwrap(verdict, 120), collapse = "\n")) +
    theme_minimal(base_size = 11) +
    theme(panel.grid = element_blank(), axis.text.x.top = element_text(face = "bold"),
          plot.title = element_text(face = "bold"), plot.caption = element_text(hjust = 0, size = 10),
          plot.title.position = "plot", plot.caption.position = "plot")
pS

### Export
The scorecard, verdict, per-tile and per-plot tables, and the main figures go to `OUT_DIR`.

In [ ]:
write_csv(score %>% mutate(role = as.character(role)), file.path(OUT_DIR, "scorecard.csv"))
writeLines(c(verdict, "", "Confounds:", CONFOUNDS, if (!is.null(stab_tests)) paste("stability:", STABILITY$caveat)),
           file.path(OUT_DIR, "verdict.txt"))
write_csv(train_sum, file.path(OUT_DIR, "A_training_by_tile.csv"))
write_csv(tile_diff, file.path(OUT_DIR, "B_map_difference_by_tile.csv"))
write_csv(pairs, file.path(OUT_DIR, "C_field_plot_pairs.csv"))
write_csv(acc_tests, file.path(OUT_DIR, "C_accuracy_tests.csv"))
if (!is.null(stab_tests)) write_csv(stab_pairs, file.path(OUT_DIR, "D_stability_by_plot.csv"))
save_fig <- function(p, name, w, h) ggsave(file.path(OUT_DIR, paste0(name, ".png")), p, width = w, height = h, dpi = 200, bg = "white")
save_fig(pS, "S_scorecard", 13, 0.45 * nrow(score) + 2)
save_fig(pA1, "A1_training_per_tile", 14, 9)
save_fig(pA2 + pA3 + pA4 + plot_layout(widths = c(2, 1.3, 1)), "A2_timing_height", 14, 4.5)
save_fig(pB1 + pB2 + pB3 + plot_layout(widths = c(1, 1.3, 1.1)), "B_map_agreement", 15, 5)
save_fig(pC1 + pC2 + plot_layout(widths = c(1.4, 1)), "C_accuracy", 15, 5.5)
save_fig(pC3, "C3_error_by_biomass", 10, 4.5)
message("Outputs in ", OUT_DIR)

## Notes and limits
- **The comparison is for one year (2020).** It is the only year with both v3.1 maps. If single-year maps are produced for more years, add their tile indices and extend §4 to each year (the multi-year advantage may vary with how good a given year's ICESat-2 acquisition was).
- **Field plots are old and few in some regions.** The paired design removes their shared error, but regions with few plot groups get wide CIs (C2). LVIS (ABoVE domain, 2017 and 2019) is the next independent reference to add for a same-year comparison; the extraction code is in `validation_v3.1_lidar_LVIS_GLiHT.ipynb`.
- **Confounds** listed in §0 are printed with the verdict. If the two runs used different `mapBoreal` commits, part of any difference is due to code, not training data.
- **Stability ≠ accuracy.** A smoother series is better only if real change is still captured; see the LVIS change test before claiming the multi-year series is better for trends.